# How to Define and Apply Contingencies
> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl/docs/Project.toml`
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.6
 >   [e30172f5] Documenter v1.17.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [98b081ad] Literate v2.21.0
 >   [bed98974] PowerNetworkMatrices v0.24.3 `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.5.0
 >   [bcd98974] PowerSystems v5.12.0
 >   [08abe8d2] PrettyTables v3.4.3
 > 
 > ```



This guide shows how to compute post-contingency PTDF rows with
`VirtualMODF` — the lazy Multiple Outage Distribution Factor matrix.
You will attach outages to a system, let them auto-register, query monitored
arcs under a contingency, and build manual modifications when you need full
control.

> *Note*
>
> There is **no dense `MODF` type**. Post-contingency factors are only
> available through `VirtualMODF`, which computes rows on demand via the
> Woodbury identity. See Flowgate Methodology for the theory.

## Prerequisites

  - `PowerNetworkMatrices.jl` and `PowerSystems.jl` installed
  - A power system model

In [ ]:
using PowerNetworkMatrices
import PowerNetworkMatrices as PNM
import PowerSystems as PSY
import PowerSystemCaseBuilder as PSB

sys = PSB.build_system(PSB.PSITestSystems, "c_sys5");

## Attach Outages to the System

Contingencies are defined as `PSY.Outage` supplemental attributes on the
components they trip. When a contingency only needs to *exist*, use a
`PSY.FixedForcedOutage` with `outage_status = 1.0` (outaged) and attach it to
each branch:

In [ ]:
for branch in PSY.get_components(PSY.ACTransmission, sys)
    outage = PSY.FixedForcedOutage(; outage_status = 1.0)
    PSY.add_supplemental_attribute!(sys, branch, outage)
end

## Build the VirtualMODF

Registration is **automatic**: the constructor scans the system for `PSY.Outage`
attributes and resolves each into a `ContingencySpec`. There is no
public `register_contingency` — construct the matrix from a system that already
carries its outages.

In [ ]:
vmodf = VirtualMODF(sys)

Inspect what was registered with `get_registered_contingencies`. It
returns a `Dict{UUID, ContingencySpec}` keyed by the source outage's UUID:

In [ ]:
contingencies = get_registered_contingencies(vmodf)

## Query a Monitored Arc Under a Contingency

Index the matrix as `vmodf[monitored_arc, spec]`. The monitored arc is an arc
tuple `(from, to)` (or its integer index); the returned value is the full
post-contingency PTDF row for that arc — one sensitivity per bus.

Pick a monitored arc and outage a *different* arc — monitoring an element that
the contingency itself outages is undefined and raises. Here the spec is built
straight from an arc with the convenience `NetworkModification`
constructor:

In [ ]:
arcs = PNM.get_arc_axis(vmodf);
monitored_arc = arcs[1];
ctg = NetworkModification(vmodf, arcs[2]);

The returned row carries one post-contingency sensitivity per bus:

In [ ]:
row = vmodf[monitored_arc, ctg]

The second index accepts three equivalent forms — the
`NetworkModification` used above, a `ContingencySpec` from the
registered set, or the original `PSY.Outage` (by
its registered UUID). All resolve to the same `NetworkModification` and
share the cached Woodbury factors, so repeated queries for one contingency across
different monitored arcs reuse work:

```julia
spec = first(values(contingencies))       # a registered ContingencySpec
vmodf[monitored_arc, spec]
vmodf[monitored_arc, spec.modification]   # its NetworkModification

branch = first(PSY.get_components(PSY.ACTransmission, sys))
outage = first(PSY.get_supplemental_attributes(branch))
vmodf[monitored_arc, outage]              # the PSY.Outage, by UUID
```

## The modification type model

Under the convenience constructor sit a few value types, layered from the smallest
unit up to the solver-ready form. It helps to know them before dropping to the
manual path:

| Type                          | Represents                                                             | Scope                      |
|:----------------------------- |:---------------------------------------------------------------------- |:-------------------------- |
| `ArcModification`     | A susceptance change on one aggregated arc, plus optional Ybus Pi-model deltas | One arc              |
| `ShuntModification`   | A diagonal admittance change on one bus                                | One bus                    |
| `NetworkModification` | A canonical, `System`-independent bundle of arc and shunt changes plus islanding status | Whole modification |
| `ContingencySpec`     | A `NetworkModification` tagged with the source `PSY.Outage` UUID | One registered contingency |

`NetworkModification` is the canonical representation: once built it holds no
reference to the source `System` and serves as the
cache key inside `VirtualMODF` (its `label` is excluded from equality, so two
physically identical modifications compare equal regardless of name).

> *Note*
>
> Partial (non-full-outage) susceptance changes are supported only on **direct and
> parallel** arcs. Series-reduced arcs and 3-winding transformer windings accept
> only a full outage of the equivalent; anything else raises an error.

## Build a Modification Manually

The convenience constructor used above (`NetworkModification(matrix, arc)`, or
`NetworkModification(matrix, branch)`) is the simplest path — it looks up the
arc's susceptance and populates the deltas for you. When you want full control,
assemble the low-level building blocks instead. An `ArcModification` is a
susceptance change on one arc (`delta_b` negative for an outage); a
`ShuntModification` is an admittance change on one bus. Both are indexed
by their **integer** position in the matrix:

```julia
arc_index = PNM.get_arc_lookup(vmodf)[(1, 4)]
arc_mod = ArcModification(arc_index, -5.0)   # Δb removes the arc's susceptance

bus_index = PNM.get_bus_lookup(vmodf)[3]
shunt_mod = ShuntModification(bus_index, ComplexF32(-0.1im))

# Combine arc and shunt changes into one modification (label, arcs, shunts, islanding)
custom = NetworkModification("arc_and_shunt", [arc_mod], [shunt_mod], false)
vmodf[monitored_arc, custom]
```

Prefer the convenience constructors over hand-built `ArcModification`
values: they compute physically consistent `delta_b` and Pi-model deltas from the
network data, which is otherwise your responsibility to get right.

## One-Shot Post-Modification Rows from a VirtualPTDF

If you already hold a `VirtualPTDF` and want a single post-modification
row without registering contingencies, use
`get_post_modification_ptdf_row`. It applies a `NetworkModification`
through the same Woodbury correction:

In [ ]:
vptdf = VirtualPTDF(sys)
varcs = PNM.get_arc_axis(vptdf);
mod = NetworkModification(vptdf, varcs[2]);
row_oneshot = get_post_modification_ptdf_row(vptdf, varcs[1], mod)

Indexing is the equivalent form — it returns the same row:

In [ ]:
isapprox(vptdf[varcs[1], mod], row_oneshot)

This function does **no caching** — each call recomputes. When querying many
monitored arcs for the *same* modification, precompute once with
`compute_woodbury_factors` and reuse via `apply_woodbury_correction`.

## Contingencies and Network Reduction

If you build the `VirtualMODF` with `network_reductions`, any branch that a
contingency outages or monitors must survive every reduction step. Outage and
monitored-component buses are auto-protected from reduction. Declare monitored
branches on the outage so their buses are kept:

```julia
monitored_line = PSY.get_component(PSY.ACTransmission, sys, "2")
PSY.set_monitored_components!(outage, [monitored_line])
```

Querying a monitored arc that was reduced away raises a clear error rather than
silently returning the base row.

## See Also

  - Public API Reference — full docstrings for `NetworkModification`,
    `ContingencySpec`, `compute_woodbury_factors`, and the rest
  - Flowgate Methodology — the Woodbury post-contingency theory